## 1. Missing Values

**Step 1 - Concept**

A missing value means the data for that customer and column was never recorded. Pandas shows this as NaN. It can happen from a skipped field, a system error while saving, or data lost during a merge/export. Missing values don't always look like NaN though - sometimes they're an empty string, a space, or a placeholder word like "unknown".

**Step 2 - Demonstrate the Concept**

**Simple example:** In a list of 10 students' marks, if one student's mark box was left empty, that's a missing value.

**Real-world example:** A customer fills out a signup form but skips the "Age" field because it wasn't marked required - that row now has a missing Age.

**Business example:** A telecom company can't correctly calculate average customer age for a marketing campaign if 50 out of 5000 customer records have no age recorded - the average would be based on incomplete data.

**AI/ML use case:** If Age is used as an input feature for a churn-prediction model and some rows have it missing, the model either can't process those rows or has to guess a placeholder value, which can bias what the model learns.

**Why it matters:** Missing values silently shrink or distort the usable data. Any statistic (mean, count, percentage) computed while ignoring this will be misleading, and any model trained on it will learn a skewed pattern.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("custumer.csv")
df.shape
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
Age,51,1.02
TotalCharges,51,1.02
MonthlyCharges,50,1.00


**Interpretation:** Age, MonthlyCharges, and TotalCharges each have about 50 missing values, roughly 1%.

Insight: Small share, but all three are numeric and matter for later analysis (correlation, target analysis), so they can't be left as is. Imputing is better than dropping (e.g. Age with median, TotalCharges from MonthlyCharges * TenureYears) - to be decided in the preprocessing sprint.

In [5]:
for col in df.select_dtypes(include="object").columns:
    empty_like = df[col].isin(["", " ", "NA", "N/A", "null", "None", "unknown", "Unknown"]).sum()
    if empty_like > 0:
        print(f"{col}: {empty_like} null-like values")
    else:
        print(f"{col}: none found")

CustomerID: none found
Gender: none found
ContractType: none found
InternetService: none found
PaymentMethod: none found
SeniorCitizen: none found
StreamingService: none found
PaperlessBilling: none found
Churn: none found


**Output:** every column prints "none found".

**Interpretation:** No hidden/disguised missing values were found in any text column.

**AI/ML Relevance:** A numeric column stored as text (from stray characters) won't throw an error - it will just be ignored or wrongly encoded by the model. Confirming there's none here avoids that hidden bug.

## 2. Duplicate Records

**Step 1 - Concept**

A duplicate row appears more than once in the dataset - from double entry, repeated exports, or merging data from multiple sources without checking for overlap.

**Step 2 - Demonstrate the Concept**

**Simple example:** A spreadsheet where row 5 and row 12 have the exact same values - same name, same marks, same everything.

**Real-world example:** A customer accidentally submits the same registration form twice because the page reloaded, creating two identical records.

**Business example:** A company merges customer lists from two regional sales teams into one file, and 15 customers who exist in both files end up appearing twice in the combined data.

**AI/ML use case:** If duplicate rows aren't removed before splitting data into training and test sets, the same customer record could land in both sets - the model then gets "tested" on data it already saw, making its accuracy look artificially high.

**Why it matters:** Duplicates inflate counts and skew any statistic based on frequency, like churn rate or average spend. They also compromise model evaluation if left in before a train/test split.

In [6]:
full_duplicates = df.duplicated().sum()
print("Fully duplicated rows:", full_duplicates)

id_duplicates = df.duplicated(subset=["CustomerID"]).sum()
print("Duplicate CustomerIDs:", id_duplicates)

Fully duplicated rows: 0
Duplicate CustomerIDs: 0


**Interpretation:** No fully duplicated rows, and no repeated CustomerID values either - every row is a unique customer.

**Insight:** Good sign for reliability - no risk of inflated counts or double-counted customers when calculating churn rate or any other aggregate statistic.

## 3. Invalid Values
**Step 1 - Concept**

Invalid values are technically present (so they pass a missing-value check) but don't make logical sense for that column - like a negative charge, or an age outside a realistic human range.

**Step 2 - Demonstrate the Concept**

**Simple example:** A "quantity purchased" column showing -3, which isn't possible - you can't buy a negative amount of something.

**Real-world example:** A billing system logs a customer's monthly charge as -500 due to a sign error during a refund entry.

**Business example:** A retail company's customer profile shows an age of "200 years" - clearly a typo, but if left in, it throws off the average age reported to management.

**AI/ML use case:** If a churn model is trained with an unrealistic Age value like 200, it can distort the correlation the model finds between age and churn, leading to wrong conclusions about which customers are actually at risk.

**Why it matters:** Invalid values pass basic checks like isnull() but still corrupt statistics and mislead a model, because the model treats them as valid, real data rather than errors.

In [7]:
print("Negative MonthlyCharges:", (df["MonthlyCharges"] < 0).sum())
print("Negative TotalCharges:", (df["TotalCharges"] < 0).sum())
print("Negative TenureYears:", (df["TenureYears"] < 0).sum())
print("Age below 18 or above 100:", ((df["Age"] < 18) | (df["Age"] > 100)).sum())
print("Negative SupportCalls:", (df["SupportCalls"] < 0).sum())

Negative MonthlyCharges: 0
Negative TotalCharges: 0
Negative TenureYears: 0
Age below 18 or above 100: 0
Negative SupportCalls: 0


In [8]:
for col in ["Gender","ContractType","InternetService","PaymentMethod","SeniorCitizen","StreamingService","PaperlessBilling","Churn"]:
    print(col, "->", sorted(df[col].dropna().unique()))

Gender -> ['Female', 'Male']
ContractType -> ['Month-to-month', 'One year', 'Two year']
InternetService -> ['DSL', 'Fiber optic', 'No']
PaymentMethod -> ['Bank transfer', 'Credit card', 'Electronic check', 'Mailed check']
SeniorCitizen -> ['No', 'Yes']
StreamingService -> ['No', 'Yes']
PaperlessBilling -> ['No', 'Yes']
Churn -> ['No', 'Yes']


**Interpretation:** No negative charges, no impossible ages, no negative tenure/support calls, and no unexpected categories.

**Insight:** Unlike missing values, invalid values are not a problem in this dataset. Worth stating clearly in the report - "I checked and found nothing" is still a real finding.

**AI/ML Relevance:** Confirms the numeric columns can be trusted for statistics like mean and standard deviation later, without distortion from a fake outlier that's really a data error.